<a href="https://colab.research.google.com/github/Grindewald1900/Galactic-Frontier/blob/master/Jupyter/Scripts/Galactic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Galactic Frontier

In [13]:
import json
import random
import matplotlib.pyplot as plt
from copy import deepcopy
from collections import defaultdict
from google.colab import drive

In [14]:
def load_cards_from_json(file_path: str):
    """
    从指定 JSON 文件加载卡牌数据，返回一个列表，每个元素是一个卡牌信息的 dict。
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        cards = json.load(f)
    return cards['characters']

def draw_cards(card_pool, num=5):
    """
    从卡牌池中随机抽取指定数量的卡牌。
    card_pool 是所有可用卡牌的列表
    num 是需要抽取的卡牌数量
    返回抽取到的卡牌列表
    """
    return random.sample(card_pool, k=num)

def simulate_battle(player_cards, enemy_cards):
    """
    一个简化的对战模拟逻辑：
    1. 两边各有若干卡牌，每张卡牌都有 attack 和 health。
    2. 对战规则（示例）：
       - 进入回合制，每回合双方各选一张卡牌来对战（此处为随机选取），进行伤害交换。
       - 伤害交换后，attack_x 对 health_y 造成 x 点伤害，attack_y 对 health_x 造成 y 点伤害。
       - 如果某张卡牌生命力降至 0 或以下，则它退出战场。
       - 重复直到一方没有卡牌为止。
    3. 最后剩余卡牌的一方为胜利方，返回胜者（"Player" 或 "Enemy"）。
    """
    # 用 deepcopy 拷贝，以免对原始卡牌造成修改
    p_cards = deepcopy(player_cards)
    e_cards = deepcopy(enemy_cards)
    # damage_record 用来记录“本场对战中各卡牌造成的总伤害”
    p_damage_record = defaultdict(int)
    e_damage_record = defaultdict(int)

    # 将卡牌放入可选池中
    while p_cards and e_cards:
        # 随机从各自卡组挑一张卡牌
        p_card = random.choice(p_cards)
        e_card = random.choice(e_cards)
        p_stats = p_card['stats']
        e_stats = e_card['stats']

        # 伤害交换
        p_stats['HP'] -= e_stats['Attack']
        e_damage_record[e_card['name']] += e_stats['Attack']
        e_stats['HP'] -= p_stats['Attack']
        p_damage_record[p_card['name']] += p_stats['Attack']

        # 检查卡牌存活
        # 如果某张卡牌血量 <= 0 则移除
        p_cards = [card for card in p_cards if card['stats']['HP'] > 0]
        e_cards = [card for card in e_cards if card['stats']['HP'] > 0]

    # 根据剩余卡牌判断胜负
    if p_cards and not e_cards:
        winner = "Player"
    elif e_cards and not p_cards:
        winner = "Enemy"
    else:
        winner = "Draw"

    return winner, p_damage_record, e_damage_record

def visualize_damage(total_damage):
    """
    使用 matplotlib 将总伤害量可视化为柱状图。
    total_damage: dict，键为卡牌名，值为累计伤害。
    """
    # 按伤害值从大到小排序，生成 (name, damage) 元组列表
    sorted_damage = sorted(total_damage.items(), key=lambda x: x[1], reverse=True)

    if not sorted_damage:
        print("没有伤害数据，无法可视化。")
        return

    # 分离出卡牌名称和伤害值，方便画图
    card_names = [item[0] for item in sorted_damage]
    damage_values = [item[1] for item in sorted_damage]

    # 创建画布
    plt.figure(figsize=(10, 6))

    # 绘制柱状图
    plt.bar(card_names, damage_values, color='skyblue')

    # 设置标题和坐标轴标签
    plt.title("各卡牌累计伤害量", fontsize=16)
    plt.xlabel("卡牌名称", fontsize=12)
    plt.ylabel("总伤害量", fontsize=12)

    # 旋转横坐标标签，防止过长时挤在一起
    plt.xticks(rotation=45, ha='right')

    # 让布局更紧凑，避免标签被截断
    plt.tight_layout()

    # 显示图像
    plt.show()

In [15]:
def main():
    # 1. 加载卡牌数据
    drive.mount('/content/drive')
    cards = load_cards_from_json("/content/drive/MyDrive/Galactic/card.json")

    # 模拟对战的次数
    num_battles = 100

    player_wins = 0
    enemy_wins = 0
    draws = 0
    player_hand = draw_cards(cards, 5)
    enemy_hand = draw_cards(cards, 5)
    # total_damage 用来记录“在所有对战中，每个卡牌造成的总伤害”
    total_damage_p = defaultdict(int)
    total_damage_e = defaultdict(int)

    print("Player Hand:")
    for card in player_hand:
        print(card)
    print("Enemy Hand:")
    for card in enemy_hand:
        print(card)

    for _ in range(num_battles):
      # 3. 进行对战模拟
      result, damage_p_in_battle, damage_e_in_battle  = simulate_battle(player_hand, enemy_hand)
      # 4. 记录结果
      if result == "Player":
          player_wins += 1
      elif result == "Enemy":
          enemy_wins += 1
      else:
          draws += 1
      # 5. 累加伤害统计
      for card_name, dmg in damage_p_in_battle.items():
          total_damage_p[card_name] += dmg
      for card_name, dmg in damage_e_in_battle.items():
          total_damage_e[card_name] += dmg

    # 输出统计结果
    print(f"经过 {num_battles} 场对战：")
    print(f"玩家胜利：{player_wins} 次")
    print(f"敌方胜利：{enemy_wins} 次")
    print(f"平局：{draws} 次")

    # 输出每个角色在所有对战中的总伤害
    print("\n=== 每个角色在所有对战中的总伤害量统计 ===")
    # 按伤害量从高到低排序，便于观察
    sorted_damage_p = sorted(total_damage_p.items(), key=lambda x: x[1], reverse=True)
    for name, dmg in sorted_damage_p:
        print(f"玩家 {name} 共造成了 {dmg} 点伤害")
    print('\n')
    sorted_damage_e = sorted(total_damage_e.items(), key=lambda x: x[1], reverse=True)
    for name, dmg in sorted_damage_e:
        print(f"敌方 {name} 共造成了 {dmg} 点伤害")
   # visualize_damage(total_damage_p)
   # visualize_damage(total_damage_e)


In [16]:
main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Player Hand:
{'name': 'Stargate Voyager · Asra', 'stats': {'Attack': 70, 'Defense': 60, 'Speed': 70, 'HP': 75, 'Crit': 10}}
{'name': 'Star-Eater Witch · Kalaishi', 'stats': {'Attack': 85, 'Defense': 50, 'Speed': 70, 'HP': 60, 'Crit': 20}}
{'name': ' Synth Abomination · Magki', 'stats': {'Attack': 80, 'Defense': 55, 'Speed': 65, 'HP': 85, 'Crit': 15}}
{'name': 'Arcane Forger · Franta', 'stats': {'Attack': 60, 'Defense': 65, 'Speed': 55, 'HP': 80, 'Crit': 10}}
{'name': 'Abyss Inquisitor · Rayel', 'stats': {'Attack': 75, 'Defense': 70, 'Speed': 55, 'HP': 75, 'Crit': 15}}
Enemy Hand:
{'name': 'Stargate Voyager · Asra', 'stats': {'Attack': 70, 'Defense': 60, 'Speed': 70, 'HP': 75, 'Crit': 10}}
{'name': 'Star-Eater Witch · Kalaishi', 'stats': {'Attack': 85, 'Defense': 50, 'Speed': 70, 'HP': 60, 'Crit': 20}}
{'name': 'Arcane Forger · Franta', 'stats': {'Attack': 60,